## Contact MinIO

In [ ]:
# %pip install boto3

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# %pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://host.docker.internal:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
    region_name="us-east-1",
    config=boto3.session.Config(signature_version="s3v4")
)

BUCKET = "bronze"

## Reading files from MinIO

In [ ]:
# response = s3.list_buckets()

# print(response)
# print(type(response))
# print(response['Buckets'])

{'ResponseMetadata': {'RequestId': '18A8F7A156BDAD2E', 'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8', 'HTTPStatusCode': 200, 'HTTPHeaders': {'accept-ranges': 'bytes', 'content-length': '364', 'content-type': 'application/xml', 'server': 'MinIO', 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'vary': 'Origin, Accept-Encoding', 'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8', 'x-amz-request-id': '18A8F7A156BDAD2E', 'x-content-type-options': 'nosniff', 'x-ratelimit-limit': '3312', 'x-ratelimit-remaining': '3312', 'x-xss-protection': '1; mode=block', 'date': 'Thu, 23 Apr 2026 11:05:25 GMT'}, 'RetryAttempts': 0}, 'Buckets': [{'Name': 'bronze', 'CreationDate': datetime.datetime(2026, 4, 16, 12, 11, 8, 184000, tzinfo=tzlocal())}], 'Owner': {'DisplayName': 'minio', 'ID': '02d6176db174dc93cb1b899f7c6078f08654445fe8cf1b6ce98d8855f66bdbf4'}}
<class 'dict'>
[{'Name': 'bronze', 'CreationDate': datetime.datetime(202

In [ ]:
# response = s3.list_objects_v2(Bucket=BUCKET)

# for obj in response.get("Contents", []):
#     print(obj["Key"])

adzuna/ingestion_timestamp=1776445291/data.json
adzuna/ingestion_timestamp=1776511963/data.json
adzuna/ingestion_timestamp=1776512923/data.json
adzuna/ingestion_timestamp=1776513218/data.json
arbeitnow/ingestion_timestamp=1776341456/data.json
arbeitnow/ingestion_timestamp=1776445291/data.json
arbeitnow/ingestion_timestamp=1776462223/data.json
arbeitnow/ingestion_timestamp=1776511410/data.json
arbeitnow/ingestion_timestamp=1776511963/data.json
arbeitnow/ingestion_timestamp=1776513218/data.json
metadata/last_run.json
reed/ingestion_timestamp=1776445291/data.json
reed/ingestion_timestamp=1776462223/data.json
reed/ingestion_timestamp=1776511410/data.json
reed/ingestion_timestamp=1776511963/data.json
reed/ingestion_timestamp=1776513218/data.json


In [ ]:
# import json
# import pandas as pd

# def load_latest_source_raw(source_name):
#     response = s3.list_objects_v2(Bucket=BUCKET)

#     files = [
#         obj["Key"] for obj in response.get("Contents", [])
#         if source_name in obj["Key"] and obj["Key"].endswith("data.json")
#     ]

#     if not files:
#         print(f"No files found for {source_name}")
#         return None

#     latest_file = sorted(files)[-1]

#     response = s3.get_object(Bucket=BUCKET, Key=latest_file)
#     data = json.loads(response["Body"].read().decode("utf-8"))

#     print(f"{source_name} loaded (RAW)")

#     return data
# adzuna_raw = load_latest_source_raw("adzuna")
# reed_raw = load_latest_source_raw("reed")
# arbeitnow_raw = load_latest_source_raw("arbeitnow")

adzuna loaded (RAW)
reed loaded (RAW)
arbeitnow loaded (RAW)


In [10]:
import os
import importlib
from scripts.ingestion import reed_client

# 1. تعيين المفتاح
os.environ["REED_API_KEY"] = "5f08f697-2811-4c29-b1dd-77db0440416b"

# 2. إجبار Python على إعادة قراءة ملف reed_client مع المتغيرات الجديدة
importlib.reload(reed_client)

# 3. التجربة الآن
try:
    reed_raw = reed_client.collect_reed()
    print(f"✅ Reed records: {len(reed_raw)}")
except Exception as e:
    print(f"❌ Reed still failing: {e}")

2026-04-23 11:17:07,593 | INFO | reed_client |  Starting Reed ingestion
2026-04-23 11:17:08,622 | INFO | reed_client | Page 1 collected: 50 jobs
2026-04-23 11:17:09,244 | INFO | reed_client | Page 2 collected: 50 jobs
2026-04-23 11:17:10,035 | INFO | reed_client | Page 3 collected: 50 jobs
2026-04-23 11:17:10,776 | INFO | reed_client | Page 4 collected: 50 jobs
2026-04-23 11:17:11,586 | INFO | reed_client | Page 5 collected: 50 jobs
2026-04-23 11:17:12,245 | INFO | reed_client | Page 6 collected: 50 jobs
2026-04-23 11:17:12,878 | INFO | reed_client | Page 7 collected: 50 jobs
2026-04-23 11:17:13,477 | INFO | reed_client | Page 8 collected: 50 jobs
2026-04-23 11:17:14,188 | INFO | reed_client | Page 9 collected: 50 jobs
2026-04-23 11:17:14,818 | INFO | reed_client | Page 10 collected: 50 jobs
2026-04-23 11:17:14,821 | INFO | reed_client | Total jobs collected: 500
✅ Reed records: 500


In [11]:
import os
import sys
import importlib
from datetime import datetime, timezone, timedelta

# ============================================================
# 1. ENVIRONMENT SETUP & PATH MAPPING
# ============================================================

# Ensure the project root is in sys.path so we can import 'scripts'
# Adjust the join path if your notebook is located deeper in the folder structure
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) 
if project_root not in sys.path:
    sys.path.append(project_root)

# Forcefully inject API keys into the environment BEFORE importing/reloading clients
os.environ["REED_API_KEY"] = "5f08f697-2811-4c29-b1dd-77db0440416b"
os.environ["ADZUNA_APP_ID"] = "e5976890"
os.environ["ADZUNA_APP_KEY"] = "fc8991de718748b4e89045351ad63fd3"

# ============================================================
# 2. CLIENT IMPORT & HOT RELOAD
# ============================================================

try:
    from scripts.ingestion import reed_client, adzuna_client, arbeitnow_client
    
    # Critical: Reload modules to ensure they pick up the os.environ changes injected above
    importlib.reload(reed_client)
    importlib.reload(adzuna_client)
    importlib.reload(arbeitnow_client)
    
    print("✅ Clients imported and environment variables refreshed successfully")
except ImportError as e:
    print(f"❌ Import error: {e}. Please check your project structure.")

# ============================================================
# 3. DATA COLLECTION FUNCTION
# ============================================================

def fetch_debug_data():
    """
    Fetches raw data from all three sources and handles exceptions locally 
    to prevent the entire pipeline from crashing.
    """
    adz_data, rd_data, arb_data = [], [], []

    # --- Testing Adzuna ---
    print("\n--- Testing Adzuna ---")
    try:
        # Note: Adzuna might still return 400 if the internal URL construction in adzuna_client is broken
        adz_data = adzuna_client.collect_adzuna()
        print(f"Adzuna Count: {len(adz_data)}")
    except Exception as e:
        print(f"Adzuna Failed: {e}")

    # --- Testing Reed ---
    print("\n--- Testing Reed ---")
    try:
        rd_data = reed_client.collect_reed()
        print(f"Reed Count: {len(rd_data)}")
    except Exception as e:
        print(f"Reed Failed: {e}")

    # --- Testing Arbeitnow ---
    print("\n--- Testing Arbeitnow ---")
    try:
        arb_data = arbeitnow_client.collect_arbeitnow()
        print(f"Arbeitnow Count: {len(arb_data)}")
    except Exception as e:
        print(f"Arbeitnow Failed: {e}")
        
    return adz_data, rd_data, arb_data

# ============================================================
# 4. EXECUTION
# ============================================================

# Execute the fetch and store results in variables for analysis
adzuna_raw, reed_raw, arbeitnow_raw = fetch_debug_data()

# Quick summary for the user
print("\n" + "="*30)
print(f"FINAL COLLECTION SUMMARY:")
print(f"Total Adzuna: {len(adzuna_raw)}")
print(f"Total Reed: {len(reed_raw)}")
print(f"Total Arbeitnow: {len(arbeitnow_raw)}")
print("="*30)

✅ Clients imported and environment variables refreshed successfully

--- Testing Adzuna ---
2026-04-23 11:19:06,890 | INFO | adzuna_client | Adzuna ingestion started
2026-04-23 11:19:06,892 | INFO | adzuna_client | Fetching Adzuna page 1
2026-04-23 11:19:10,098 | INFO | adzuna_client | Fetching Adzuna page 2
2026-04-23 11:19:14,142 | INFO | adzuna_client | Fetching Adzuna page 3
2026-04-23 11:19:17,924 | INFO | adzuna_client | Fetching Adzuna page 4
2026-04-23 11:19:23,561 | INFO | adzuna_client | Fetching Adzuna page 5
2026-04-23 11:19:27,857 | INFO | adzuna_client | Fetching Adzuna page 6
2026-04-23 11:19:37,278 | INFO | adzuna_client | Fetching Adzuna page 7
2026-04-23 11:19:41,071 | INFO | adzuna_client | Fetching Adzuna page 8
2026-04-23 11:19:44,856 | INFO | adzuna_client | Fetching Adzuna page 9
2026-04-23 11:19:49,017 | INFO | adzuna_client | Fetching Adzuna page 10
2026-04-23 11:19:53,025 | INFO | adzuna_client | Adzuna collected 500 jobs
Adzuna Count: 500

--- Testing Reed --

In [12]:
import sys
sys.path.append(r"C:\Users\Admin\OneDrive\Desktop\Projects\JobIntelligent-Data-Platform")
from scripts.processing.cleaner import clean_adzuna_data, clean_reed_data, clean_arbeitnow_data
adzuna_clean = clean_adzuna_data(adzuna_raw)
reed_clean = clean_reed_data(reed_raw)
arbeitnow_clean = clean_arbeitnow_data(arbeitnow_raw)

2026-04-23 11:20:39,711 | INFO | data_cleaner | [Adzuna] Initial records: 500
2026-04-23 11:20:39,712 | INFO | data_cleaner | [Adzuna] Final records: 500
2026-04-23 11:20:39,714 | INFO | data_cleaner | [Adzuna] Dropped records: 0
2026-04-23 11:20:39,715 | INFO | data_cleaner | [Reed] Starting cleaning process


/home/jovyan/scripts/processing/cleaner.py:48: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(str(text), "html.parser")
/home/jovyan/scripts/processing/cleaner.py:48: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(str(text), "html.parser")


2026-04-23 11:20:39,732 | INFO | data_cleaner | [Reed] Loaded 500 raw records
2026-04-23 11:20:39,735 | INFO | data_cleaner | [Reed] Column mapping applied
2026-04-23 11:20:40,480 | INFO | data_cleaner | [Reed] Text, URL, and location cleaned
2026-04-23 11:20:40,489 | INFO | data_cleaner | [Reed] Company names normalized
2026-04-23 11:20:40,499 | INFO | data_cleaner | [Reed] Salary fields cleaned
2026-04-23 11:20:40,506 | INFO | data_cleaner | [Reed] Currency normalized
2026-04-23 11:20:40,523 | INFO | data_cleaner | [Reed] Dates parsed
2026-04-23 11:20:40,543 | INFO | data_cleaner | [Reed] Initial records: 500
2026-04-23 11:20:40,545 | INFO | data_cleaner | [Reed] After cleaning: 500
2026-04-23 11:20:40,546 | INFO | data_cleaner | [Reed] Dropped records: 0
2026-04-23 11:20:40,548 | INFO | data_cleaner | [Reed] Cleaning process completed successfully


/home/jovyan/scripts/processing/cleaner.py:48: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(str(text), "html.parser")


2026-04-23 11:20:43,857 | INFO | data_cleaner | Initial records: 324
2026-04-23 11:20:43,861 | INFO | data_cleaner | Final records: 320
2026-04-23 11:20:43,863 | INFO | data_cleaner | Dropped records: 4


/home/jovyan/scripts/processing/cleaner.py:48: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(str(text), "html.parser")


In [24]:
# Display the first cleaned job from each source
print("--- First Cleaned Adzuna Job ---")
if not adzuna_clean.empty:
    display(adzuna_clean.head(1))
else:
    print("Adzuna data is empty.")

print("\n--- First Cleaned Reed Job ---")
if not reed_clean.empty:
    display(reed_clean.head(5))
else:
    print("Reed data is empty.")

print("\n--- First Cleaned Arbeitnow Job ---")
if not arbeitnow_clean.empty:
    display(arbeitnow_clean.head(1))
else:
    print("Arbeitnow data is empty.")
    

--- First Cleaned Adzuna Job ---


,job_id,job_title,company_name,job_description,tags,location,salary_min,salary_max,posted_date,expires_date,contract_type,currency,remote,job_url
0,5691511995,Data Automation internship,Procter & Gamble,Job Location PARIS GO-ASNIERES-SUR-SEINE Job D...,Unknown,"Asnières-sur-Seine, Nanterre",NaN,NaN,2026-04-06 22:25:01+00:00,None,full_time,None,False,https://www.adzuna.fr/land/ad/5691511995



--- First Cleaned Reed Job ---


,job_id,job_title,company_name,job_description,tags,location,salary_min,salary_max,posted_date,expires_date,contract_type,currency,remote,job_url
0,56787192,Primary Teacher,Teaching Personnel,"Primary Teacher Full-Time, Part-Time Supply Op...",NaN,PO160UD,NaN,NaN,NaT,NaT,NaN,UNKNOWN,False,https://www.reed.co.uk/jobs/primary-teacher/56...
1,56787191,Site Manager,Hays Specialist Recruitment,Long-Term Temporary Site Manager - Commercial ...,NaN,Suffolk,NaN,NaN,NaT,NaT,NaN,UNKNOWN,False,https://www.reed.co.uk/jobs/site-manager/56787191
2,56787187,Truss Operative,Pearson Whiffin Recruitment,Production / Truss Operative Temp to Perm Sout...,NaN,Ashford,12.71,12.71,NaT,NaT,NaN,GBP,False,https://www.reed.co.uk/jobs/truss-operative/56...
3,56787186,Warehouse/ Stock controller,Adecco,Warehouse Operative / Stock Controller FLT Lic...,NaN,Kettering,NaN,NaN,NaT,NaT,NaN,UNKNOWN,False,https://www.reed.co.uk/jobs/warehouse-stock-co...
4,56787184,Legal Assistant - Conveyancing,Tn Recruits,Are you a Legal Assistant or Secretary looking...,NaN,TN401EY,27000.00,29000.00,NaT,NaT,NaN,GBP,False,https://www.reed.co.uk/jobs/legal-assistant-co...



--- First Cleaned Arbeitnow Job ---


,job_id,job_title,company_name,job_description,tags,location,salary_min,salary_max,posted_date,expires_date,contract_type,currency,remote,job_url
0,buroassistenz-im-ingenieurburo-kassel-372373,Büroassistenz im Ingenieurbüro m/w/d,DÖRING Beratende Ingenieure GmbH,Als Büroassistenz arbeitest du eng mit unseren...,"Building, Supply, Safety Services Engineering",Kassel,NaN,NaN,2026-04-23,NaN,NaN,NaN,False,https://www.arbeitnow.com/jobs/companies/dorin...


In [18]:
# Displaying only column names for each cleaned source
print("Adzuna Columns:", adzuna_clean.columns.tolist())
print("Reed Columns:", reed_clean.columns.tolist())
print("Arbeitnow Columns:", arbeitnow_clean.columns.tolist())

Adzuna Columns: ['job_id', 'job_title', 'company_name', 'job_description', 'tags', 'location', 'salary_min', 'salary_max', 'posted_date', 'expires_date', 'contract_type', 'currency', 'remote', 'job_url']
Reed Columns: ['job_id', 'job_title', 'company_name', 'job_description', 'tags', 'location', 'salary_min', 'salary_max', 'posted_date', 'expires_date', 'contract_type', 'currency', 'remote', 'job_url']
Arbeitnow Columns: ['job_id', 'job_title', 'company_name', 'job_description', 'tags', 'location', 'salary_min', 'salary_max', 'posted_date', 'expires_date', 'contract_type', 'currency', 'remote', 'job_url']


## Currency Exploration

In [22]:

print(f"{'='*20} CURRENCY STANDARDIZATION CHECK {'='*20}")

for df, name in zip([adzuna_clean, reed_clean, arbeitnow_clean], ['Adzuna', 'Reed', 'Arbeitnow']):
    if 'currency' in df.columns:
        # Check unique values and frequency
        counts = df['currency'].value_counts(dropna=False)
        print(f"\n[Source: {name}]")
        print(counts)
    else:
        print(f"\n[Source: {name}] Column 'currency' is missing.")

# Technical Depth: We look for (gbp vs GBP) or symbols (£ vs GBP)

==================== CURRENCY STANDARDIZATION CHECK ====================

[Source: Adzuna]
currency
None    500
Name: count, dtype: int64

[Source: Reed]
currency
GBP        296
UNKNOWN    204
Name: count, dtype: int64

[Source: Arbeitnow]
currency
NaN    320
Name: count, dtype: int64


In [28]:
import pandas as pd
import requests
import numpy as np

def fetch_rates_to_mad():
    """
    Fetches global rates and calculates conversion factor to MAD.
    """
    try:
        # We get all rates relative to USD as a standard baseline
        url = "https://api.exchangerate-api.com/v4/latest/USD"
        response = requests.get(url)
        data = response.json()
        
        rates = data['rates']
        mad_rate_to_usd = rates.get('MAD', 10.0) # 1 USD = X MAD
        
        # Calculate: How many MAD per 1 unit of foreign currency
        # Formula: (1 / Rate_to_USD) * MAD_to_USD
        conversion_map = {curr: (1/rate) * mad_rate_to_usd for curr, rate in rates.items()}
        
        return conversion_map
    except Exception as e:
        print(f"⚠️ API Error: {e}. Using static MAD fallbacks.")
        # Fallbacks: 1 GBP ≈ 12.5 MAD | 1 EUR ≈ 10.8 MAD
        return {'GBP': 12.5, 'EUR': 10.8, 'USD': 10.1, 'MAD': 1.0}

def standardize_and_convert_to_mad(df, source_name):
    df = df.copy()
    mad_rates = fetch_rates_to_mad()
    
    # --- Step 1: Currency Inference ---
    # Ensure missing currencies are filled based on source origin
    if 'currency' in df.columns:
        if source_name.lower() == 'reed':
            df['currency'] = df['currency'].fillna('GBP')
        elif source_name.lower() in ['adzuna', 'arbeitnow']:
            df['currency'] = df['currency'].fillna('EUR')
        else:
            df['currency'] = df['currency'].fillna('MAD')

    # --- Step 2: Row-level Conversion ---
    def convert_to_mad(row, col_name):
        curr = str(row.get('currency', 'MAD')).upper()
        val = pd.to_numeric(row[col_name], errors='coerce')
        
        if pd.isna(val) or curr not in mad_rates:
            return val 
        
        # Multiply by the calculated MAD factor
        return round(val * mad_rates[curr], 2)

    # Apply to all salary columns with the new _mad suffix
    for salary_col in ['salary_min', 'salary_max']:
        if salary_col in df.columns:
            df[f'{salary_col}_mad'] = df.apply(lambda r: convert_to_mad(r, salary_col), axis=1)
    
    df['currency_standardized'] = 'MAD'
    return df

# ============================================================
# EXECUTION: Standardizing our sources to Moroccan Dirham
# ============================================================
adzuna_std = standardize_and_convert_to_mad(adzuna_clean, 'Adzuna')
reed_std = standardize_and_convert_to_mad(reed_clean, 'Reed')
arbeitnow_std = standardize_and_convert_to_mad(arbeitnow_clean, 'Arbeitnow')

# Quick check on Reed data
if not reed_std.empty:
    print("\n--- Sample of Reed Salaries in MAD ---")
    display(reed_std[['job_title', 'salary_min', 'currency', 'salary_min_mad']].head(10))


--- Sample of Reed Salaries in MAD ---


,job_title,salary_min,currency,salary_min_mad
0,Primary Teacher,NaN,UNKNOWN,NaN
1,Site Manager,NaN,UNKNOWN,NaN
2,Truss Operative,12.71,GBP,158.66
3,Warehouse/ Stock controller,NaN,UNKNOWN,NaN
4,Legal Assistant - Conveyancing,27000.00,GBP,337044.53
5,Design Engineer Mechanical / Vintage Aircraft,50000.00,GBP,624156.55
6,Managed Care Consultant - Dorset,32000.00,GBP,399460.19
7,Electrician Maintenance,46800.00,GBP,584210.53
8,Building Safety Manager PERM,NaN,UNKNOWN,NaN
9,Building Safety Manager PERM,NaN,UNKNOWN,NaN


## Contract Type Exploration

In [27]:
print(f"{'='*20} CONTRACT TYPE MAPPING CHECK {'='*20}")

for df, name in zip([adzuna_clean, reed_clean, arbeitnow_clean], ['Adzuna', 'Reed', 'Arbeitnow']):
    if 'contract_type' in df.columns:
        unique_contracts = df['contract_type'].value_counts(dropna=False)
        print(f"\n[Source: {name}]")
        print(unique_contracts)
    else:
        print(f"\n[Source: {name}] Column 'contract_type' is missing.")

# Technical Depth: Adzuna and Reed often differ in naming (e.g., 'permanent' vs 'full_time')

==================== CONTRACT TYPE MAPPING CHECK ====================

[Source: Adzuna]
contract_type
None         203
permanent    137
full_time    126
contract      32
part_time      2
Name: count, dtype: int64

[Source: Reed]
contract_type
NaN    500
Name: count, dtype: int64

[Source: Arbeitnow]
contract_type
NaN    320
Name: count, dtype: int64


## Location Standardization Check

In [ ]:
print(f"{'='*20} LOCATION GRANULARITY CHECK {'='*20}")

for df, name in zip([adzuna_clean, reed_clean, arbeitnow_clean], ['Adzuna', 'Reed', 'Arbeitnow']):
    if 'location' in df.columns:
        # Show top 10 most frequent locations to identify patterns
        top_locations = df['location'].value_counts(dropna=False).head(10)
        print(f"\n[Source: {name}] - Top 10 Locations:")
        print(top_locations)
    else:
        print(f"\n[Source: {name}] Column 'location' is missing.")

# Technical Depth: Look for strings like 'Remote' vs structured addresses

## Temporal Analysis

In [ ]:
#  (Date formats and ranges)
print(f"{'='*20} TEMPORAL STANDARDIZATION CHECK {'='*20}")

for df, name in zip([adzuna_clean, reed_clean, arbeitnow_clean], ['Adzuna', 'Reed', 'Arbeitnow']):
    if 'posted_date' in df.columns:
        col = df['posted_date']
        
        # Checking the technical data type (Object/String vs Datetime64)
        dtype_info = col.dtype
        
        # Discovering the temporal range (How fresh is the data?)
        min_date = col.min()
        max_date = col.max()
        
        print(f"\n[Source: {name}]")
        print(f"   - Data Type: {dtype_info}")
        print(f"   - Date Range: From {min_date} to {max_date}")
        
        # Peek at the format of the first non-null record
        sample_val = col.dropna().iloc[0] if not col.dropna().empty else "N/A"
        print(f"   - Sample Format: {sample_val}")
    else:
        print(f"\n[Source: {name}] Column 'posted_date' is missing.")

# Technical Depth: If we find 'object' types, we MUST cast them using pd.to_datetime()
# during the final standardization phase to ensure sorting works correctly.

## Identifier Integrity & Global Overlap Analysis

In [ ]:

print(f"{'='*20} ID INTEGRITY & COLLISION CHECK {'='*20}")

# Tracking IDs across all sources to detect cross-platform conflicts
id_pools = {}

for df, name in zip([adzuna_clean, reed_clean, arbeitnow_clean], ['Adzuna', 'Reed', 'Arbeitnow']):
    if 'job_id' in df.columns:
        # 1. Internal Uniqueness: Are there duplicates within the same source?
        internal_dupes = df['job_id'].duplicated().sum()
        id_pools[name] = set(df['job_id'].astype(str))
        
        print(f"\n[Source: {name}]")
        print(f"   - Internal Duplicates: {internal_dupes}")
        print(f"   - Sample ID Pattern: {df['job_id'].iloc[0] if not df.empty else 'N/A'}")

# 2. Global Overlap Check: The 'Collision' Test
print("\n" + "-"*30)
print("🔍 Cross-Source Collision Analysis:")

sources = list(id_pools.keys())
for i in range(len(sources)):
    for j in range(i + 1, len(sources)):
        src1, src2 = sources[i], sources[j]
        common = id_pools[src1].intersection(id_pools[src2])
        if common:
            print(f"   ⚠️ CONFLICT: {len(common)} IDs overlap between {src1} and {src2}!")
        else:
            print(f"   ✅ Clean: No ID overlaps between {src1} and {src2}.")

# Technical Depth: If overlaps exist, our Standardization Strategy must involve 
# 'Namespacing' (e.g., prefixing IDs with 'reed_' or 'adz_') to guarantee global uniqueness.

In [19]:
import pandas as pd
import numpy as np

# ============================================================
# STANDARD SCHEMA (Intermediate layer before canonical)
# ============================================================

STANDARD_COLUMNS = [
    "job_id",
    "title",
    "company",
    "description",
    "location",
    "salary_min",
    "salary_max",
    "posted_date",
    "expires_date",
    "contract_type",
    "currency",
    "remote",
    "url",
    "tags"
]


# ============================================================
# SAFE rename utility
# ============================================================

def safe_rename(df, mapping: dict):
    """Rename only existing columns (avoid KeyErrors / partial mapping issues)"""
    mapping = {k: v for k, v in mapping.items() if k in df.columns}
    return df.rename(columns=mapping)


# ============================================================
# BASE NORMALIZER (shared logic)
# ============================================================

def base_standardize(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensures all standard columns exist and align structure.
    """

    df = df.copy()

    # Ensure all columns exist
    for col in STANDARD_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

    # Keep only standard structure
    return df[STANDARD_COLUMNS]


# ============================================================
# ADZUNA STANDARDIZER
# ============================================================

def standardize_adzuna(df: pd.DataFrame) -> pd.DataFrame:

    mapping = {
        "id": "job_id",
        "title": "title",
        "company.display_name": "company",
        "description": "description",
        "location.display_name": "location",
        "salary_min": "salary_min",
        "salary_max": "salary_max",
        "created": "posted_date",
        "contract_type": "contract_type",
        "redirect_url": "url",
        "category.label": "tags"
    }

    df = safe_rename(df, mapping)

    df["source"] = "adzuna"

    return base_standardize(df)


# ============================================================
# REED STANDARDIZER
# ============================================================

def standardize_reed(df: pd.DataFrame) -> pd.DataFrame:

    mapping = {
        "jobId": "job_id",
        "jobTitle": "title",
        "employerName": "company",
        "jobDescription": "description",
        "locationName": "location",
        "minimumSalary": "salary_min",
        "maximumSalary": "salary_max",
        "date": "posted_date",
        "expirationDate": "expires_date",
        "jobUrl": "url",
        "currency": "currency"
    }

    df = safe_rename(df, mapping)

    df["source"] = "reed"

    return base_standardize(df)


# ============================================================
# ARBEITNOW STANDARDIZER
# ============================================================

def standardize_arbeitnow(df: pd.DataFrame) -> pd.DataFrame:

    mapping = {
        "slug": "job_id",
        "title": "title",
        "company_name": "company",
        "description": "description",
        "location": "location",
        "url": "url",
        "created_at": "posted_date",
        "tags": "tags",
        "remote": "remote"
    }

    df = safe_rename(df, mapping)

    df["source"] = "arbeitnow"

    return base_standardize(df)


# ============================================================
# MAIN ENTRY POINT (auto-detect source)
# ============================================================

def standardize_data(df: pd.DataFrame, source: str) -> pd.DataFrame:

    source = source.lower()

    if source == "adzuna":
        return standardize_adzuna(df)

    elif source == "reed":
        return standardize_reed(df)

    elif source == "arbeitnow":
        return standardize_arbeitnow(df)

    else:
        raise ValueError(f"Unknown source: {source}")

In [35]:
adzuna_std = standardize_data(pd.json_normalize(adzuna_raw["data"]), "adzuna")
reed_std = standardize_data(pd.json_normalize(reed_raw["data"]), "reed")
arbeitnow_std = standardize_data(pd.json_normalize(arbeitnow_raw["data"]), "arbeitnow")